# LangChain + Pinecone RAG — Open-Source LLM Edition

## Cell 1 — Install dependencies

In [11]:
import sys

!{sys.executable} -m pip install -U \
    langchain \
    langchain-community \
    langchain-core \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-ollama \
    langchain-pinecone \
    sentence-transformers \
    pinecone \
    pypdf \
    unstructured[pdf]

  Using cached pinecone-8.1.2-py3-none-any.whl.metadata (14 kB)



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Cell 2 — Imports

In [12]:
import os
import glob
import getpass

from pinecone import Pinecone, ServerlessSpec

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import OllamaLLM
from langchain_pinecone import PineconeVectorStore

from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

print("All imports successful!")

All imports successful!


In [13]:
from dotenv import load_dotenv
import os

load_dotenv()  # reads .env file automatically

# Verify keys loaded (shows True/False, never the actual key)
print("PINECONE_API_KEY loaded:", bool(os.environ.get("PINECONE_API_KEY")))
print("GROQ_API_KEY loaded:", bool(os.environ.get("GROQ_API_KEY")))

PINECONE_API_KEY loaded: True
GROQ_API_KEY loaded: True


## Cell 3 — Load PDF documents

In [14]:
# Update this path to your local PDF folder
pdf_folder = r"C:\Users\psai2\OneDrive\Desktop\Gen-AI\LangChain_with_pinecone\innominds-20260224T140939Z-1-001\innominds"

pdf_files = glob.glob(f"{pdf_folder}/**/*.pdf", recursive=True)

docs = []
for pdf_file in pdf_files:
    try:
        loader = PyPDFLoader(pdf_file)
        docs.extend(loader.load())
        print(f"Loaded: {pdf_file}")
    except Exception as e:
        print(f"Error loading {pdf_file}: {e}")

print(f"\nTotal pages loaded: {len(docs)}")
if docs:
    print(f"Preview: {docs[0].page_content[:200]}")

Loaded: C:\Users\psai2\OneDrive\Desktop\Gen-AI\LangChain_with_pinecone\innominds-20260224T140939Z-1-001\innominds\Annexure 2 MSDS.pdf
Loaded: C:\Users\psai2\OneDrive\Desktop\Gen-AI\LangChain_with_pinecone\innominds-20260224T140939Z-1-001\innominds\GMC benefit manual (2022-2023).pdf
Loaded: C:\Users\psai2\OneDrive\Desktop\Gen-AI\LangChain_with_pinecone\innominds-20260224T140939Z-1-001\innominds\HR Policy and Procedure Manual.pdf
Loaded: C:\Users\psai2\OneDrive\Desktop\Gen-AI\LangChain_with_pinecone\innominds-20260224T140939Z-1-001\innominds\Innominds Policies on Use of Open Source March 2019.pdf
Loaded: C:\Users\psai2\OneDrive\Desktop\Gen-AI\LangChain_with_pinecone\innominds-20260224T140939Z-1-001\innominds\Patents &  Patent Incentive policy.pdf
Loaded: C:\Users\psai2\OneDrive\Desktop\Gen-AI\LangChain_with_pinecone\innominds-20260224T140939Z-1-001\innominds\Policy & Bench guidelines.pdf
Loaded: C:\Users\psai2\OneDrive\Desktop\Gen-AI\LangChain_with_pinecone\innominds-20260224T140939Z-1-0

## Cell 4 — Split documents into chunks

In [15]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=20
)
split_docs = text_splitter.split_documents(docs)
print(f"Total chunks: {len(split_docs)}")

Total chunks: 344


## Cell 5 — Load HuggingFace Embeddings (free, local, no API key)

Model: `all-MiniLM-L6-v2` — 80MB, fast, dimension = **384**  
Downloads once and is cached locally after that.

In [16]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},      # change to "cuda" if you have a GPU
    encode_kwargs={"normalize_embeddings": True}
)

# Verify
test_vec = embeddings.embed_query("Hello world")
print(f"Embedding dimension: {len(test_vec)}")  # expects 384

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dimension: 384


## Cell 6 — Initialize Pinecone and create index

In [17]:
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

index_name = "innominds"
EMBEDDING_DIM = 384

existing_indexes = [idx["name"] for idx in pc.list_indexes()]

if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=EMBEDDING_DIM,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print(f"Index '{index_name}' created.")
else:
    print(f"Index '{index_name}' already exists.")

index = pc.Index(index_name)
print(index.describe_index_stats())

Index 'innominds' already exists.
{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 1032}},
 'total_vector_count': 1032,
 'vector_type': 'dense'}


## Cell 7 — Embed & upsert documents into Pinecone

> ⚠️ Run this **only once** (or when you want to refresh the index).  
> If the index is already populated, skip to Cell 8.

In [18]:
docsearch = PineconeVectorStore.from_documents(
    documents=split_docs,
    embedding=embeddings,
    index_name=index_name
)
print("Documents embedded and upserted into Pinecone successfully!")

Documents embedded and upserted into Pinecone successfully!


## Cell 8 — Load existing index (skip Cell 7 if already populated)

In [19]:
# Use this if the index is already populated — no re-embedding needed
docsearch = PineconeVectorStore(
    index=index,
    embedding=embeddings
)
print("Loaded existing Pinecone index.")

Loaded existing Pinecone index.


## Cell 9 — Load Ollama LLM

Make sure Ollama is running (`ollama serve` in terminal or it runs as a background service).  

| Model | Size | Notes |
|---|---|---|
| `mistral` | 4.1 GB | Great balance of speed & quality |
| `llama3.2` | 2.0 GB | Fastest, good quality |
| `llama3.1` | 4.7 GB | Best quality |
| `gemma2:2b` | 1.6 GB | Lightest option |

Pull your model first: `ollama pull mistral`

In [20]:
import sys
!{sys.executable} -m pip install langchain-groq -q


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
from dotenv import load_dotenv
load_dotenv(override=True)  # force reload

from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    api_key=os.environ["GROQ_API_KEY"] 
)

print(llm.invoke([HumanMessage(content="Say hello in one sentence.")]).content)

Hello, how can I assist you today?


## Cell 10 — Build the RAG chain (LCEL)

In [26]:
retriever = docsearch.as_retriever(search_kwargs={"k": 3})

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are an expert HR assistant for Innominds. Your job is to answer employee questions 
accurately based strictly on the company policy documents provided.

INSTRUCTIONS:
- Answer ONLY from the context provided below
- Be specific and include exact details like numbers, dates, and eligibility criteria
- If the context does not contain enough information to answer, say: 
  "This information is not available in the provided policy documents."
- Do not make up or assume any information
- If the answer spans multiple policies, mention which policy each part comes from
- Keep the answer clear and structured

CONTEXT FROM POLICY DOCUMENTS:
{context}

EMPLOYEE QUESTION:
{question}

ANSWER:
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain ready!")

RAG chain ready!


## Cell 11 — Helper function to ask questions

In [27]:
def ask(question: str):
    """Run a question through the RAG pipeline and print the answer with sources."""
    # Get source docs separately for display
    source_docs = retriever.invoke(question)

    # Get the answer
    answer = qa_chain.invoke(question)

    print(f"Q: {question}")
    print(f"\nA: {answer}")
    print(f"\nSources:")
    for doc in source_docs:
        src = doc.metadata.get('source', 'unknown').split('\\')[-1]  # just the filename
        page = doc.metadata.get('page', '?')
        print(f"  - {src}, page {page}")
    print("-" * 60)

## Cell 12 — Ask questions!

In [28]:
ask("how are maternity leaves handled?")

Q: how are maternity leaves handled?

A: Based on the provided policy documents, here's the information regarding maternity leaves:

**Eligibility Criteria:**
To be eligible for maternity leave, an employee must have completed at least 80 days of continuous service in the 12 months preceding the date of maternity leave (Section 3, Maternity Benefit Policy).

**Duration of Leave:**
The maximum duration of maternity leave is 26 weeks, which includes 8 weeks of paid leave and 18 weeks of unpaid leave (Section 4, Maternity Benefit Policy).

**Notice Period:**
An employee must give at least 30 days' notice to the HR department before taking maternity leave (Section 5, Maternity Benefit Policy).

**Leave Application:**
The employee must submit a leave application in the prescribed format, which is available on the company's intranet, at least 30 days prior to the expected date of delivery (Section 6, Maternity Benefit Policy).

**Medical Certificate:**
The employee must submit a medical cert

In [29]:
ask("what will be covered under group medical insurance plan?")

Q: what will be covered under group medical insurance plan?

A: Based on the provided policy documents, I am unable to answer the question as there is no information available regarding the group medical insurance plan. 

However, I can suggest that you refer to the Innominds Employee Handbook or the Benefits Policy Document for more information on the group medical insurance plan.

If you have access to the actual policy documents, please provide them, and I will be happy to assist you with your question.

Alternatively, if you have any other questions that can be answered based on the provided context, please feel free to ask.

Sources:
  - GMC benefit manual (2022-2023).pdf, page 17.0
  - GMC benefit manual (2022-2023).pdf, page 18.0
  - GMC benefit manual (2022-2023).pdf, page 18.0
------------------------------------------------------------


In [30]:
ask("what are the cabin allocation guidelines?")

Q: what are the cabin allocation guidelines?

A: This information is not available in the provided policy documents.

Sources:
  - HR Policy and Procedure Manual.pdf, page 58.0
  - HR Policy and Procedure Manual.pdf, page 58.0
  - HR Policy and Procedure Manual.pdf, page 58.0
------------------------------------------------------------


In [31]:
ask("how many meeting rooms are available in Waverock 2.1?")

Q: how many meeting rooms are available in Waverock 2.1?

A: This information is not available in the provided policy documents.

Sources:
  - HR Policy and Procedure Manual.pdf, page 1.0
  - HR Policy and Procedure Manual.pdf, page 1.0
  - HR Policy and Procedure Manual.pdf, page 1.0
------------------------------------------------------------


In [32]:
ask("what is the key message for developers?")

Q: what is the key message for developers?

A: The key message for developers is as follows:

- Do not use software marked as “confidential” or containing other restrictive markings (e.g., “proprietary”) from any third party, and information from Innominds employees who have non-public third-party information from prior jobs (e.g., competitive information). 
- Confidential information often contains restrictions on both use and disclosure.
- Explicit or implicit recording of company name or your name in the code is not allowed.
- Explicitly and implicitly storing customer code or Innominds code in a code repository that is NOT approved by Innominds and Customer is not allowed.
- Clean / wipe Engineers Laptop between projects to remove any proprietary & Confidential content belonging to previous project (customers).

This information is available in the "Key Message for Developers" section of the policy document "Proprietary and Confidential to Innominds 2019 5".

Sources:
  - Innominds